In [ ]:
import pandas as pd

In [2]:
X_train = pd.read_csv('../data/processed/train.csv')

In [5]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 381109 entries, 0 to 381108
Data columns (total 14 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   age                       381109 non-null  float64
 1   driving_license           381109 non-null  int64  
 2   region_code               381109 non-null  float64
 3   previously_insured        381109 non-null  int64  
 4   vehicle_damage            381109 non-null  int64  
 5   annual_premium            381109 non-null  float64
 6   policy_sales_channel      381109 non-null  float64
 7   vintage                   381109 non-null  float64
 8   gender_female             381109 non-null  int64  
 9   gender_male               381109 non-null  int64  
 10  vehicle_age_1_to_2_years  381109 non-null  int64  
 11  vehicle_age_over_2_years  381109 non-null  int64  
 12  vehicle_age_under_1_year  381109 non-null  int64  
 13  response                  381109 non-null  i

---

# API tests

extra libs

In [33]:
import pandas as pd
import json
import requests
from pathlib import Path
import joblib

schemas.py

In [2]:
from pydantic import BaseModel, ConfigDict
from typing import List

class PredictionRequest(BaseModel):
    model_config = ConfigDict(extra="allow")

class PredictionItem(BaseModel):
    prediction: int
    probability: float

class PredictionResponse(BaseModel):
    predictions: List[PredictionItem]

predict.py

In [34]:
from insurance_classifier.config import load_yaml_config

config = load_yaml_config('../config/base.yaml')

def predict_request(model, data: PredictionRequest, pipeline):
    df = pd.DataFrame(data)

    # Preprocessing

    df1 = pipeline.feature_engineering(df) 
    df2 = pipeline.data_preparation(df1)

    # Prediction
    preds = model.predict(df2)
    probas = model.predict_proba(df2)[:,1]

    results = [
    {
        "prediction": int(pred),
        "probability": float(proba)
    }
    for pred, proba in zip(preds, probas)
    ]

    return PredictionResponse(predictions=results)

pipeline.py

In [35]:
class PreprocessingPipeline(object):

    def __init__(self, config):
        self.config = config
        parameters_path = Path(config['paths']['parameters_dir'])

        self.age_scaler =                   joblib.load(parameters_path / config['artifacts']['age_scaler']) 
        self.annual_premium_scaler =        joblib.load(parameters_path / config['artifacts']['annual_premium_scaler'])
        self.vintage_scaler =               joblib.load(parameters_path / config['artifacts']['vintage_scaler'])
        self.region_code_encoder =          joblib.load(parameters_path / config['artifacts']['region_code_encoder'])               
        
    def feature_engineering(self, df):  
          
        # Change column names to snakecase
        df.columns = [col.lower().replace(' ','_') for col in df.columns]

        # Rename vehicle_age categories
        df['vehicle_age'] = df['vehicle_age'].map({'> 2 Years': 'over_2_years',
                                                            '1-2 Year': '1_to_2_years',
                                                            '< 1 Year': 'under_1_year'
                                                            })

        # Change string entries to snakecase
        for col in df.select_dtypes(exclude=['int64','float64', 'datetime64[ns]']).columns:
            df[col] = df[col].str.lower()

        return df

    def data_preparation(self, df):
        # One-Hot encoding
        df = pd.get_dummies(df, columns= ['gender'], prefix= 'gender', dtype=int)

        # Target encoding (James-Stein) 
        df['region_code'] = self.region_code_encoder.transform(X= df[['region_code']])

        # Label encoding
        df['vehicle_damage'] = df['vehicle_damage'].map({'yes':1,'no':0})

        # One-Hot encoding
        df = pd.get_dummies(df, columns= ['vehicle_age'], prefix= 'vehicle_age', dtype=int)

        # Frequency encoding
        fe_policy_sales_channel = df['policy_sales_channel'].value_counts(normalize=True)
        df['policy_sales_channel'] = df['policy_sales_channel'].map(fe_policy_sales_channel)

        # Rescaling -----
        df['age'] = self.age_scaler.transform(df[['age']].values)
        df['vintage'] = self.vintage_scaler.transform(df[['vintage']].values)

        # Standardization -----
        df['annual_premium'] = self.annual_premium_scaler.transform(df[['annual_premium']].values)

        return df[self.config['features']['selected_features']]
    

main.py

In [13]:
from contextlib import asynccontextmanager

from fastapi import FastAPI
import joblib
import uvicorn

from pathlib import Path

ml_model = {}
model_config = load_yaml_config("../config/base.yaml")

# context manager (model load)
@asynccontextmanager
async def lifespan(app:FastAPI):
    ml_model["xgb_v01"] = joblib.load(Path(model_config["paths"]["model_dir"]) / "xgb_v01.joblib")
    ml_model["preprocessing"] = PreprocessingPipeline(model_config)
    yield
    ml_model.clear()

app = FastAPI(title="Insurance Classifier API", 
              version="v1", 
              lifespan=lifespan)

@app.get("/health")
async def health_check():
    return {"status": "ok"}

@app.post("/predict", response_model=PredictionRequest)
def predict_endpoint(request: PredictionRequest):
    model = ml_model["xgb_v01"]
    pipeline = ml_model["preprocessing"]

    return predict_request(model, request, pipeline=pipeline)


# if __name__ == "__main__":
#     uvicorn.run("app.main:app", host="0.0.0.0", port=8000)


In [40]:
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv')

df_test = pd.concat([X_test, y_test], axis=1)

real_test = pd.read_csv('../data/raw/test.csv')

In [41]:
real_test.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage
0,381110,Male,25,1,11.0,1,< 1 Year,No,35786.0,152.0,53
1,381111,Male,40,1,28.0,0,1-2 Year,Yes,33762.0,7.0,111
2,381112,Male,47,1,28.0,0,1-2 Year,Yes,40050.0,124.0,199
3,381113,Male,24,1,27.0,1,< 1 Year,Yes,37356.0,152.0,187
4,381114,Male,27,1,28.0,1,< 1 Year,No,59097.0,152.0,297


In [42]:
X_test.head()

,age,driving_license,region_code,previously_insured,vehicle_damage,annual_premium,policy_sales_channel,vintage,gender_female,gender_male,vehicle_age_1_to_2_years,vehicle_age_over_2_years,vehicle_age_under_1_year
0,0.015385,1,0.125168,1,0,-0.590037,0.057146,0.214533,1,0,0,0,1
1,0.538462,1,0.090622,0,1,0.402809,0.209127,0.318339,0,1,1,0,0
2,0.323077,1,0.125168,0,1,-1.622853,0.209127,0.276817,1,0,1,0,0
3,0.092308,1,0.117303,0,0,-1.622853,0.010194,0.892734,1,0,0,0,1
4,0.476923,1,0.120237,0,1,0.199941,0.194157,0.882353,0,1,1,0,0


In [43]:
df_sample = real_test.sample(10, random_state=42)

In [44]:
payload = json.dumps(df_sample.to_dict(orient='records'))

In [45]:
payload

'[{"id": 491724, "Gender": "Female", "Age": 36, "Driving_License": 1, "Region_Code": 29.0, "Previously_Insured": 0, "Vehicle_Age": "1-2 Year", "Vehicle_Damage": "Yes", "Annual_Premium": 2630.0, "Policy_Sales_Channel": 156.0, "Vintage": 56}, {"id": 468163, "Gender": "Female", "Age": 43, "Driving_License": 1, "Region_Code": 21.0, "Previously_Insured": 0, "Vehicle_Age": "1-2 Year", "Vehicle_Damage": "Yes", "Annual_Premium": 32003.0, "Policy_Sales_Channel": 124.0, "Vintage": 162}, {"id": 419450, "Gender": "Male", "Age": 22, "Driving_License": 1, "Region_Code": 28.0, "Previously_Insured": 1, "Vehicle_Age": "< 1 Year", "Vehicle_Damage": "No", "Annual_Premium": 31631.0, "Policy_Sales_Channel": 160.0, "Vintage": 128}, {"id": 384310, "Gender": "Male", "Age": 25, "Driving_License": 1, "Region_Code": 50.0, "Previously_Insured": 1, "Vehicle_Age": "< 1 Year", "Vehicle_Damage": "No", "Annual_Premium": 31831.0, "Policy_Sales_Channel": 152.0, "Vintage": 262}, {"id": 385006, "Gender": "Male", "Age": 33

In [38]:
df_convert = pd.DataFrame(json.loads(payload))

In [39]:
df_convert.head()

,age,driving_license,region_code,previously_insured,vehicle_damage,annual_premium,policy_sales_channel,vintage,gender_female,gender_male,vehicle_age_1_to_2_years,vehicle_age_over_2_years,vehicle_age_under_1_year
0,0.030769,1,0.107496,1,0,0.304338,0.353663,0.577855,0,1,0,0,1
1,0.338462,1,0.150772,0,1,1.122086,0.209127,0.366782,0,1,1,0,0
2,0.046154,1,0.090622,1,0,0.916894,0.353663,0.837370,1,0,0,0,1
3,0.369231,1,0.150772,1,1,0.630484,0.004894,0.643599,1,0,1,0,0
4,0.092308,1,0.150772,1,0,0.688347,0.353663,0.816609,1,0,0,0,1


In [27]:
url = "http://localhost:8000/predict"

In [29]:
response = requests.post(url, data=payload, headers={"Content-Type": "application/json"})

In [ ]:
response.json()